# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

Colab의 GPU 환경에서 개발되었습니다.
- 런타임 - 런타임 유형 변경 - GPU로 변경(T4 GPU 등)



# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [1]:
# 1. 코랩이 제멋대로 깔아둔 변종 파이토치와 찌꺼기들을 무자비하게 삭제합니다.
%pip uninstall -y torch torchvision torchaudio flash-attn

# 2. Flash-Attention 완제품 공장과 정확히 일치하는 '공식 순정 파이토치 2.5.1'을 설치합니다. (약 30초 소요)
%pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124

# 3. 필수 라이브러리 설치
%pip -q install "transformers>=4.49.0, <5.0.0" "accelerate>=1.1.0" "peft>=0.14.0" datasets "pandas==2.2.2" "pillow<12.0.0" --upgrade
%pip install matplotlib

# 4. 바탕이 완벽해졌으므로, 10초짜리 완제품(Wheel)을 강제 주입합니다. 이제 절대 튕겨내지 않습니다.
%pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.0.post2/flash_attn-2.7.0.post2+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl

import torch
print(f"현재 GPU: {torch.cuda.get_device_name()} / VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}GB")
print("PyTorch:", torch.__version__)
print("궁극의 1분 컷 환경 준비를 마쳤습니다.")

Found existing installation: torch 2.5.1+cu124
Uninstalling torch-2.5.1+cu124:
  Successfully uninstalled torch-2.5.1+cu124
Found existing installation: torchvision 0.20.1+cu124
Uninstalling torchvision-0.20.1+cu124:
  Successfully uninstalled torchvision-0.20.1+cu124
Found existing installation: torchaudio 2.5.1+cu124
Uninstalling torchaudio-2.5.1+cu124:
  Successfully uninstalled torchaudio-2.5.1+cu124
Found existing installation: flash-attn 2.7.0.post2
Uninstalling flash-attn-2.7.0.post2:
  Successfully uninstalled flash-attn-2.7.0.post2
Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached https://download.pytorch.org/whl/cu124/torch-2.5.1%2Bcu124-cp312-cp312-linux_x86_64.whl (908.2 MB)
  Using cached https://download-r2.pytorch.org/whl/cu124/torchvision-0.20.1%2Bcu124-cp312-cp312-linux_x86_64.whl (7.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu124/torchaudio-2.5.1%2Bcu124-cp312-cp312-linux_x86_64.whl (3.4 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!unzip '/content/drive/MyDrive/2026-ssafy-ai-15-2.zip' -d '/content/'

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: /content/train/train_0074.jpg  
  inflating: /content/train/train_0075.jpg  
  inflating: /content/train/train_0076.jpg  
  inflating: /content/train/train_0077.jpg  
  inflating: /content/train/train_0078.jpg  
  inflating: /content/train/train_0079.jpg  
  inflating: /content/train/train_0080.jpg  
  inflating: /content/train/train_0081.jpg  
  inflating: /content/train/train_0082.jpg  
  inflating: /content/train/train_0083.jpg  
  inflating: /content/train/train_0084.jpg  
  inflating: /content/train/train_0085.jpg  
  inflating: /content/train/train_0086.jpg  
  inflating: /content/train/train_0087.jpg  
  inflating: /content/train/train_0088.jpg  
  inflating: /content/train/train_0089.jpg  
  inflating: /content/train/train_0090.jpg  
  inflating: /content/train/train_0091.jpg  
  inflating: /content/train/train_0092.jpg  
  inflating: /content/train/train_0093.jpg  
  inflating: /content/train/train_0094.jpg  
  inflating: /conte

In [7]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

Torch version: 2.5.1+cu124
CUDA version: 12.4
cuDNN version: 90100


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 colab에서 구글 드라이브를 마운트하여 사용합니다.

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습
    - 구글 드라이브 마운트 : drive()

# 라이브러리, 데이터, 설정

In [8]:
import os, re, math, random, collections
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Dict, List, Any
from torchvision import transforms
from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    get_cosine_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output

# 시드 고정
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 학습 및 모델 설정 (H100에 맞춘 스펙업)
MODEL_ID = "Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled-v2"
MAX_PIXELS = 1024 * 1024  # H100의 VRAM을 활용하여 모델의 시력을 극대화합니다 (기존 720x720 대비 대폭 상향).
EPOCHS = 5
BATCH_SIZE = 4    # 80GB VRAM에 맞추어 한 번에 학습할 이미지 매수를 늘립니다.
GRAD_ACCUM = 4    # BATCH_SIZE가 늘어났으므로 누적 스텝을 줄여 학습 속도를 올립니다.
LEARNING_RATE = 2e-5

device = "cuda" if torch.cuda.is_available() else "cpu"

def get_multi_crops(img, crop_size=448):
    w, h = img.size
    crop_w, crop_h = min(w, crop_size), min(h, crop_size)
    crops = [img]
    crops.append(img.crop((0, 0, crop_w, crop_h)))
    crops.append(img.crop((w - crop_w, 0, w, crop_h)))
    crops.append(img.crop((0, h - crop_h, crop_w, h)))
    crops.append(img.crop((w - crop_w, h - crop_h, w, h)))
    return crops

# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [9]:
train_transforms = transforms.Compose([
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.RandomHorizontalFlip(p=0.5),
])

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
processor.tokenizer.padding_side = 'right' # 학습 시에는 정답 토큰 예측을 위해 우측 패딩을 유지합니다.

# H100 특화: 양자화(BitsAndBytes)를 제거하고 bfloat16과 FlashAttention-2로 직결합니다.
base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="flash_attention_2",
    trust_remote_code=True
)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

base_model.enable_input_require_grads()
base_model.gradient_checkpointing_enable()

# LoRA 설정 강화: r값과 alpha값을 높여 모델의 더 많은 지능을 파인튜닝합니다.
lora_config = LoraConfig(
    r=128,
    lora_alpha=256,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


ImportError: /usr/local/lib/python3.12/dist-packages/flash_attn_2_cuda.cpython-312-x86_64-linux-gnu.so: undefined symbol: _ZN3c104cuda29c10_cuda_check_implementationEiPKcS2_ib

# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [4]:
# [Cell 수정본] 0.97 도달을 위한 '강제 사고(CoT) 영어 프롬프트'
SYSTEM_INSTRUCT = (
    "You are a highly logical visual forensic expert performing visual question answering. "
    "Analyze the provided visual area meticulously. Eliminate incorrect options based on even the smallest visual cues. "
    "Output exactly: 'Reasoning: [your logic]' followed by 'Answer: [a/b/c/d]'."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"Question: {question}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "Analyze this area step-by-step:"
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [ ]:
# 커스텀 데이터셋
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}

# 데이터 콜레이터
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
            images.append(img)

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if self.train:
            enc["labels"] = enc["input_ids"].clone()

        return enc


# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [ ]:
# 검증용 데이터 분리
split = int(len(train_df)*0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

# VQAMCDataset 형태로 변환
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# 데이터로더
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=DataCollator(processor, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False, collate_fn=DataCollator(processor, True), num_workers=0)

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
from tqdm.auto import tqdm

model = model.to(device)
GRAD_ACCUM = 4

# 옵티마이저, 학습 스케줄러
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
num_training_steps = 1 * math.ceil(len(train_loader)/GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, int(num_training_steps*0.03), num_training_steps)

# 스케일러
scaler = torch.cuda.amp.GradScaler(enabled=True)

# 학습 루프
global_step = 0
for epoch in range(1):
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
            running = 0.0

    model.eval()
    val_loss = 0.0
    val_steps = 0
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f"[Epoch {epoch+1}] valid loss {val_loss/val_steps:.4f}")
    model.train()

# 모델 저장
SAVE_DIR = "/content/qwen2_5_vl_3b_lora"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)


# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [6]:
import os, re, collections, torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
# 주인님의 원본 코드에 있던 클래스를 그대로 사용합니다.
from transformers import AutoModelForVision2Seq, AutoProcessor
from peft import PeftModel

# 1. 설정 및 경로 (주인님의 원본 파일 기준)
MODEL_ID = "OpenDataArena/MMFineReason-8B"
# 학습 코드가 끝난 후 저장된 LoRA 가중치 폴더의 이름을 적어주세요.
LORA_PATH = "./mmfinereason_vqa_lora_epoch_1"
MAX_PIXELS = 720 * 720
INFERENCE_BATCH_SIZE = 32
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. 모델 및 프로세서 독립 로드
print(f"MMFineReason-8B 모델과 LoRA 가중치를 로드합니다: {LORA_PATH}")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

# 10시간 허공 연산을 방지하는 유일한 통제줄입니다.
processor.tokenizer.padding_side = 'left'

# H100의 속도를 100% 뽑아내기 위해 bfloat16을 적용합니다.
base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

# 베이스 모델 위에 주인님이 학습시킨 가중치를 강제로 씌웁니다.
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

# 3. 필수 함수 정의
def get_multi_crops(img, crop_size=448):
    w, h = img.size
    crop_w, crop_h = min(w, crop_size), min(h, crop_size)
    crops = [img]
    crops.append(img.crop((0, 0, crop_w, crop_h)))
    crops.append(img.crop((w - crop_w, 0, w, crop_h)))
    crops.append(img.crop((0, h - crop_h, crop_w, h)))
    crops.append(img.crop((w - crop_w, h - crop_h, w, h)))
    return crops

def build_mc_prompt(question, a, b, c, d):
    return (
        f"Question: {question}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "Analyze this area step-by-step:"
    )

SYSTEM_INSTRUCT = (
    "You are a highly logical visual forensic expert performing visual question answering. "
    "Analyze the provided visual area meticulously. Eliminate incorrect options based on even the smallest visual cues. "
    "Output exactly: 'Reasoning: [your logic]' followed by 'Answer: [a/b/c/d]'."
)

# 4. 데이터 로드 및 추론 시작
print("데이터를 로드하고 추론을 시작합니다.")
test_df = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")

predictions = []
pseudo_labels = []

with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
    for i in tqdm(range(0, len(test_df), INFERENCE_BATCH_SIZE), desc="Batched Inference Progress"):
        batch_df = test_df.iloc[i:i+INFERENCE_BATCH_SIZE]

        all_views = []
        all_text_prompts = []

        for _, row in batch_df.iterrows():
            img_path = row["path"]
            if not os.path.exists(img_path): continue

            original_img = Image.open(img_path).convert("RGB")
            original_img.thumbnail((720, 720))

            views = get_multi_crops(original_img)
            user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

            for view in views:
                messages = [
                    {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
                    {"role": "user", "content": [{"type": "image", "image": view}, {"type": "text", "text": user_text}]}
                ]
                text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

                all_views.append(view)
                all_text_prompts.append(text_prompt)

        if not all_views: continue

        inputs = processor(
            text=all_text_prompts, images=all_views, padding=True, return_tensors="pt", max_pixels=MAX_PIXELS
        ).to(device)

        # 모델의 무의미한 장문 생성을 억제하여 병목을 막습니다.
        generated_ids = model.generate(
            **inputs, max_new_tokens=64, temperature=0.3, do_sample=True, top_p=0.9
        )

        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        output_texts = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)

        for j, (_, row) in enumerate(batch_df.iterrows()):
            start_idx = j * 5
            end_idx = start_idx + 5
            row_outputs = output_texts[start_idx:end_idx]

            votes = []
            for output_text in row_outputs:
                output_text_lower = output_text.strip().lower()
                match = re.search(r'answer:\s*([abcd])', output_text_lower)
                pred = match.group(1) if match else (re.findall(r'[abcd]', output_text_lower)[-1] if re.findall(r'[abcd]', output_text_lower) else 'a')
                votes.append(pred)

            vote_counts = collections.Counter(votes)
            final_answer = vote_counts.most_common(1)[0][0]
            predictions.append(final_answer)

            if vote_counts[final_answer] == 5:
                pseudo_labels.append({"path": row["path"], "question": row["question"],
                                      "a": row["a"], "b": row["b"], "c": row["c"], "d": row["d"], "answer": final_answer})

# 5. 결과 저장 및 포맷팅
test_df["answer"] = predictions
# sample_submission의 ID 순서에 맞춰 안전하게 덮어씌웁니다.
sample_sub["answer"] = sample_sub["id"].map(test_df.set_index("id")["answer"])

sample_sub.to_csv("submission.csv", index=False)
pd.DataFrame(pseudo_labels).to_csv("pseudo_labels.csv", index=False)

print(f"최종 submission.csv 파일이 생성되었습니다. 의사 라벨 개수: {len(pseudo_labels)}")

MMFineReason-8B 모델과 LoRA 가중치를 로드합니다: ./mmfinereason_vqa_lora_epoch_5


preprocessor_config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

ValueError: Can't find 'adapter_config.json' at './mmfinereason_vqa_lora_epoch_5'

In [ ]:
# 모델 응답 예시
print(output_text)